# Predicción de rotación de empleados

Proyecto educativo de Big Data y analítica de Recursos Humanos. Integra datos, prepara variables, compara clasificadores y aplica los resultados a una empresa hipotética y sus áreas de negocio.

## 1. Resumen ejecutivo

El análisis integra información laboral con una encuesta sintética, realiza controles de calidad y ETL con PySpark, y compara Regresión Logística, Random Forest y MLP mediante validación cruzada anidada. Regresión Logística lidera ROC-AUC (0,823), PR-AUC (0,575) y F1 evaluado (0,542). Con su umbral de referencia de 0,706, las puntuaciones OOF reúnen 251 señales: 110 en Ventas, 131 en Investigación y Desarrollo y 10 en Recursos Humanos. Ventas tiene la tasa observada más alta (20,6 %); los registros con horas extra presentan 30,5 %, frente a 10,4 % sin ellas. Estos datos ofrecen prioridades concretas para los equipos de negocio.

## 2. Objetivo y alcance

- **Objetivo:** comparar modelos y describir patrones de rotación para orientar un plan de negocio.
- **Tarea:** clasificar `Attrition` y analizar resultados por departamento y características laborales.
- **Métricas:** ROC-AUC, PR-AUC, F1 evaluado, exactitud y tasas observadas.
- **Aplicación:** ordenar diagnósticos, acordar pilotos y seguir indicadores de retención.

## 3. Datos y calidad

El archivo `WA_Fn-UseC_-HR-Employee-Attrition.csv` contiene 1.470 registros y 35 columnas del conjunto conocido como IBM HR Analytics Employee Attrition & Performance. `encuesta_clima.csv` se genera con `scripts/generate_synthetic_survey.py`: cinco escalas de 1 a 5 y semilla fija 42, sin usar la etiqueta `Attrition`. Las columnas `survey_*` se excluyen del modelado.

`scripts/check_data_quality.py` verifica la clave, duplicados, nulos y categorías de Attrition. Consulta los detalles en `docs/DATA_SOURCE.md`.

Docker Compose reúne Jupyter, Spark Master y PostgreSQL. El ETL utiliza PySpark en modo local (`local[*]`) desde Jupyter. El servicio monta datos, scripts, notebooks, resultados y las figuras del panel. PostgreSQL almacena características, métricas y predicciones para su consulta y seguimiento.

## 5. Preparación de datos

`scripts/etl_attrition.py` normaliza columnas y textos, convierte las escalas de la encuesta y une los ficheros por `EmployeeNumber`. La unión izquierda conserva los 1.470 registros principales. El proceso construye `attrition_label`, `overtime_flag`, `income_yearly` y `tenure_ratio`, y guarda un Parquet integrado de 43 columnas.

`scripts/train_ml.py` excluye identificadores, constantes, etiquetas y variables sintéticas. `income_yearly` y `overtime_flag` también quedan fuera del entrenamiento porque derivan de variables ya presentes.

## 6. Construcción y evaluación de modelos

El pipeline transforma las variables numéricas con `StandardScaler` y las categóricas con `OneHotEncoder`, ajustando el preprocesamiento dentro de cada partición.

- **Regresión Logística:** regularización L2 y ponderación balanceada.
- **Random Forest:** 300 árboles y ponderación `balanced_subsample`.
- **MLP:** capas de 64 y 32 neuronas, activación ReLU y parada temprana.

La evaluación canónica usa cinco folds externos estratificados. En cada entrenamiento externo, una validación interna de tres folds selecciona el umbral; el fold externo queda reservado para la evaluación. Después, cada pipeline final se ajusta con los 1.470 registros para generar sus artefactos de operación.

## 7. Resultados comparativos

| Modelo | ROC-AUC | PR-AUC | F1 evaluado | Umbral de referencia (mediana) | Exactitud a 0,5 | F1 a 0,5 |
|---|---:|---:|---:|---:|---:|---:|
| Regresión Logística | 0,823 | 0,575 | 0,542 | 0,706 | 0,748 | 0,472 |
| Random Forest | 0,803 | 0,540 | 0,523 | 0,257 | 0,855 | 0,242 |
| MLP | 0,814 | 0,559 | 0,522 | 0,316 | 0,868 | 0,458 |

Regresión Logística lidera las tres métricas evaluadas. El F1 evaluado promedia los cinco folds externos con el umbral seleccionado dentro de cada entrenamiento. La mediana de esos cinco umbrales sirve de referencia común para contar señales OOF; la tabla también incluye exactitud y F1 al corte fijo de 0,5.

## 8. Lectura de las visualizaciones

- **Puntuaciones:** el histograma agrupa 1.470 probabilidades OOF de Regresión Logística en intervalos de 0,10. Hay 370 valores por debajo de 0,10 y 747 (50,8 %) por debajo de 0,30.
- **Departamentos:** Ventas registra 20,6 % de rotación observada (92/446), Recursos Humanos 19,0 % (12/63) e Investigación y Desarrollo 13,8 % (133/961).
- **Horas extra:** 30,5 % (127/416) frente a 10,4 % (110/1.054) sin horas extra.
- **Señales OOF:** al corte 0,706, Ventas reúne 110, Investigación y Desarrollo 131 y Recursos Humanos 10.
- **Factores asociados:** entre los coeficientes de mayor magnitud aparecen Director/a de investigación (−1,161), Representante de ventas (+1,024), horas extra: sí (+0,968) y viajes frecuentes (+0,878).

## 9. Conclusiones y recomendaciones para dirección

Regresión Logística es el modelo de referencia por sus mejores ROC-AUC, PR-AUC y F1 evaluado. Ventas combina la tasa observada más alta con 110 señales OOF, por lo que conviene iniciar allí el diagnóstico comercial: revisar viajes, objetivos y distribución de carga. Investigación y Desarrollo reúne 133 salidas observadas y 131 señales; su revisión por equipo y puesto puede orientar el acompañamiento hacia los grupos con más casos. La diferencia entre las tasas observadas con y sin horas extra respalda revisar planificación, turnos y cobertura. En Recursos Humanos, la revisión se dimensiona a partir de 12 salidas y 10 señales.

Estas prioridades organizan el análisis de negocio por tasa, volumen de salidas y señales; permiten definir pilotos concretos y responsables para cada área.

5. Mostrar por separado las tasas observadas, las señales OOF, las métricas OOF y los coeficientes asociados.

## 11. Panel interactivo en HTML

El archivo `docs/panel/Panel_Rotacion_Empleados_interactivo.html` permite seleccionar Regresión Logística, Random Forest o MLP y consultar sus métricas de evaluación. También presenta tasas observadas por departamento y horas extra, señales OOF de Regresión Logística al umbral de referencia y factores asociados. Las métricas OOF, las tasas observadas y las señales se muestran por separado para conservar su significado. Descarga el HTML y ábrelo en un navegador; no hace falta ejecutar Docker para consultar el panel.

## 12. Reproducción del flujo

Desde la raíz del repositorio, ejecuta estos pasos en orden:

1. Inicia el entorno: `docker compose up -d --build`.
2. Genera la encuesta con `scripts/generate_synthetic_survey.py`, semilla 42.
3. Revisa calidad con `scripts/check_data_quality.py` y construye el Parquet con `scripts/etl_attrition.py`.
4. Ejecuta `docker compose run --rm jupyter python /scripts/train_ml.py --input /data/processed/employee_attrition.parquet --model all`.
5. Carga características con `scripts/load_features_to_pg.py` y persiste predicciones de LogReg, RF y MLP con `scripts/persist_results.py`.
6. Regenera figuras con `docker compose run --rm jupyter python /scripts/render_dashboard_previews.py`.
7. Abre los notebooks para recorrer el análisis; `02_Modelado_Baseline` y `03_Modelado_DL` son prácticas exploratorias con sus propios splits.

Las métricas canónicas y las predicciones OOF se guardan en `output/metrics/` y `output/dashboard/`. La ejecución reproducible de GitHub Actions está enlazada en `docs/RESULTADOS_FINALES.md`.

## Documentación relacionada

- [Resultados finales verificados](../docs/RESULTADOS_FINALES.md)
- [Procedencia de datos](../docs/DATA_SOURCE.md)
- [Protocolo de modelado](../docs/MODELING_PROTOCOL.md)
- [Panel interactivo en HTML](../docs/panel/Panel_Rotacion_Empleados_interactivo.html)